<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weekend 9 — Multi-Agent Systems

**The big idea this session:** *Why a team of specialised agents beats one do-everything agent — and how to wire them together.*

---


### What you'll be able to do
By the end you can:
- **Explain** in plain English why one agent doing many jobs underperforms a team of focused agents.
- **Describe** the three coordination patterns — **sequential, parallel, supervisor** — and pick the right one.
- **Demonstrate** how agents talk by passing outputs (text / JSON) into the next agent's input.
- **Implement** a real multi-agent pipeline with the Claude API.
- **Architect** a production multi-agent system and name its failure points, costs, and safeguards.

### How each section is structured
| Block | What it gives you |
|---|---|
| **Six questions** | What / why / how / where / when to use / when to avoid |
| **Enterprise · Architect · Oracle DBA** | How the idea shows up in a real company, a real system, and in database terms you already know |
| **Remember this** | One interview-ready line |
| **Don't mix these up** | Quick wrong vs. right corrections |
| **Interview Q** | A likely question + a model answer |
| **Quick check + See it** | A question you answer, then real code you run |

# Foundations — the words you need first

Before patterns and code, five plain definitions. Keep these handy.

- **Agent** — *one* Claude call given a **role** (a job description) plus a task. In this session an agent = **role + its own memory + an input to output**.
- **Role (`system` prompt)** — the text telling Claude *who it is and how to behave* ("You are a meticulous market researcher"). The main place specialisation happens.
- **Multi-agent system** — several agents wired so one agent's **output** becomes another's **input**.
- **Handoff** — passing one agent's output text into the next agent's prompt. Its *format* (prose vs. JSON) decides reliability.
- **Coordination pattern** — *how* the agents are wired: **sequential** (a line), **parallel** (side by side), or **supervisor** (a manager decides).

> **Mental model:** a multi-agent system is just a **company org chart made of Claude calls**. Each person (agent) has one job; work flows between desks (handoffs); a structure (pattern) decides who does what, when.
>
> **Oracle DBA view:** think of each agent like a **stored procedure** — a defined job with clear inputs and outputs. The `system` prompt is its responsibility (what it's allowed to do), each agent runs in its **own session** (no shared state unless you pass it), and wiring agents together is like building an **ETL pipeline**: each stage hands a clean result to the next.


# Run-along setup

The code cells let you *see* every idea. Add your key to Colab Secrets (key icon, left sidebar) as **`MY_API_KEY`**, then run the two setup cells below. You do this once.

> **Plain Jupyter (not Colab)?** Replace the key line with `os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."` read from your environment — never hardcode a real key in a shared notebook.


In [ ]:
# Cell 1 - install the Anthropic SDK (the official Claude library)
!pip install anthropic

In [ ]:
# Cell 2 - set your key and create the client (run once)
from google.colab import userdata
import os, json
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # cheap + fast - ideal for many-call agent labs
print("Ready")

### Our one reusable helper: `ask_agent`
We use this tiny function all weekend. It turns **any role** into a working agent: `role` becomes the system prompt (the job description), and `task` is the user message for *this* agent only — a **fresh, isolated memory** each time.

In [ ]:
def ask_agent(role, task, max_tokens=400):
    """Run ONE agent: 'role' = its job (system prompt), 'task' = what to do now.
    Each call has its own fresh messages list, so agents do not share memory."""
    reply = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=role,                                   # WHO the agent is
        messages=[{"role": "user", "content": task}],  # WHAT to do right now
    )
    return reply.content[0].text                       # the agent's answer (plain text)

print("ask_agent ready")

---
# Section 1 - Why One Agent Isn't Enough

**What is it?** The core problem this session solves: when you ask a *single* agent to do many jobs in one prompt, quality drops on every job.

**Why does it matter?** Quality is the product. An "okay-at-everything" assistant loses to a team that is "excellent at each step" — and clients feel the difference immediately.

**How does it work (what goes wrong inside the model)?** A single prompt saying *"research **and** analyse **and** write **and** follow these 12 rules"* causes two failures:
- **Context dilution** — the model's attention spreads thin across competing instructions, so each gets less focus.
- **Instruction blur** — rules for one job bleed into another, so nothing is followed cleanly.

**Where is it used?** Everywhere humans split work: a newsroom (reporters, editors, fact-checkers), a kitchen (prep, line, plating), a bank's deal team (research, risk, client memo).

**When should I use a team?** When the task has **distinct steps or skills** that each deserve full attention.

**When should I avoid it?** When the task is small and single-skill — one focused agent is then cheaper, faster, *and* just as good (the over-splitting trap, Section 6).


### The overload problem, three ways
1. **One employee, three jobs.** Research a market, crunch the numbers, *and* write the board report in one sitting — all three come out "okay", none "excellent".
2. **One chef, five dishes at once.** Everything plates lukewarm and late.
3. **One prompt, many instructions.** The model spreads attention thin and blurs the rules.

**Oracle DBA analogy:** it's one DBA asked to run backups, tune SQL, *and* patch security all in the same hour — each task gets done at "okay", none at "excellent". A real shop splits this across a backup DBA, a performance DBA, and a security DBA. Same idea here.

The fix is the same: **give each job its own specialist.**

### Enterprise Perspective
A bank building a **credit-memo assistant** first tried one giant prompt: pull the borrower's financials, score the risk, *and* write the memo. The risk scoring was shallow and the memo missed required disclosures. Splitting into **Researcher to Risk Analyst to Memo Writer** lifted each step to specialist quality and made the output auditable — every stage produced a record a compliance officer could inspect. Outcome: faster memos that actually pass review.

### Instructor Notes
Best analogy: *a one-person startup vs. a team.* The solo founder ships everything at 70%; the team ships each piece at 95%. Most common confusion: "but it's the same model — how can splitting help?" Answer: **same brain, but one job at a time instead of five at once.** Attention is the scarce resource, not knowledge.

> **Remember this:** *One agent doing many jobs isn't limited by what the model knows — it's limited by attention. Splitting the work gives each step the model's full focus.*

### Don't mix these up
- WRONG: "A bigger prompt = a smarter agent."  RIGHT: A bigger multi-job prompt usually means *more dilution*, not more intelligence.
- WRONG: "Multi-agent helps because each agent is a smarter model."  RIGHT: It's the *same* model — the win is focus, isolation, and clean handoffs.
- WRONG: "Always split into many agents."  RIGHT: Split only when there are genuinely distinct steps.

### Interview question
**Q (beginner):** "Why would three focused agents beat one agent given the same task and model?"
**A:** A single multi-job prompt causes *context dilution* (attention spread thin) and *instruction blur* (rules bleed between tasks). Giving each job its own role and clean input means each step gets full focus and a predictable output — higher quality per step and far easier to debug.

## Quick Check 1
> **Q:** A teammate says "just put everything in one big prompt — it's simpler." Technical downside?
> **A:** **Context dilution + instruction blur** — competing instructions share attention and rules leak across tasks, so every job degrades. Focused agents keep each task's context clean.

### See it: one overloaded agent vs. one focused step
Run both. The overloaded prompt is broad but shallow; the focused step goes deep on its slice. That depth gap *is* the argument for teams.

In [ ]:
# A - one agent told to do THREE jobs at once (watch it stay shallow)
overloaded = client.messages.create(
    model=MODEL, max_tokens=400,
    messages=[{"role": "user", "content":
        "Research whether Zepto should expand 10-minute delivery to Indian tier-2 cities, "
        "analyse the risks, AND write a final 1-paragraph recommendation - all now."}],
)
print("OVERLOADED (3 jobs in 1 prompt):\n", overloaded.content[0].text)

In [ ]:
# B - the SAME research slice, but as one focused specialist (watch it go deep)
focused = ask_agent(
    "You are a meticulous market researcher. Output 5 concise, factual bullet points only.",
    "Key factors for Zepto expanding 10-minute delivery into Indian tier-2 cities?",
)
print("FOCUSED (research step only):\n", focused)

---
# Section 2 - What Is a Multi-Agent System?

**What is it?** Several Claude calls, each given a **different role**, wired so one's output feeds the next. No special framework required.

**Why does it matter?** Once you see an "agent" as just *role + call*, you can build any team with one tiny helper and a few lines of plain Python. No magic.

**How does it work?** An agent here is **three things**: (1) a **role** set by the `system` prompt, (2) its **own memory** — a separate `messages` list isolated from others, and (3) an **input to output** — it takes a task and returns text the next agent can use.

**Where is it used?** Every "AI workflow" product under the hood: support copilots, research assistants, report generators.

**When to use / avoid?** Use the moment you have two or more distinct roles. Avoid framework overhead when a single `ask_agent` call already does the job.


### `system` prompt vs. `user` message - the key distinction
- **`system`** = *who the agent is* and *how it behaves* (stable across the task) -> the **role**.
- **`messages`** = *the specific thing to do right now* (changes each call) -> the **task**.

Same machinery, different role text -> a completely different specialist. **Specialisation lives in the `system` prompt.**

### Oracle DBA Perspective
This is just like having **one database engine** serve very different jobs depending on *who connects*. Connect as a read-only reporting user and you get cautious, read-only behaviour; connect as a power user and you get something very different — **same engine, different role**. Here the `system` prompt is that "connected role." You don't need a new model (a new engine) to get new behaviour; you just change the role string. It's cheap, transparent, and easy to review — exactly what you'd want before putting anything into production.

> **Remember this:** *An agent is just role + memory + input/output. To make two opposite specialists from one model, change only the `system` prompt — nothing else.*
>
> **Oracle DBA view:** the same database engine behaves completely differently depending on the **schema/role** you connect as. Same model, different `system` prompt = same engine, different connected user.

### Don't mix these up
- WRONG: "Different agents need different models."  RIGHT: Usually the *same* model with different roles. Model choice is about cost/capability, not identity.
- WRONG: "Agents automatically remember each other's conversations."  RIGHT: Each agent has its **own** `messages` list. Info travels only if *you* pass it.

### Interview question
**Q (intermediate):** "How do you make two agents behave differently with the same model and code?"
**A:** Change only the `system` prompt — the role text. Model, helper function, and user message stay identical; the role is the specialisation lever.

## Quick Check 2
> **Q:** You want two agents with opposite behaviours from the same model. What do you change?
> **A:** Only the **`system` prompt** (the role). Model, code, and user message can stay the same.

### See it: same question, two roles, two personalities
The *only* thing changing is the role string. Watch the output flip — proof that specialisation lives in `system`.

In [ ]:
question = "Should Zepto enter Jaipur?"

cautious = ask_agent("You are a risk-averse analyst. Emphasise dangers and reasons to wait.", question)
bold     = ask_agent("You are a growth-focused strategist. Emphasise upside and speed.", question)

print("CAUTIOUS ROLE:\n", cautious[:350], "\n")
print("BOLD ROLE:\n", bold[:350])

---
# Section 3 - The Three Coordination Patterns

**What is it?** The three ways to wire agents together. Almost every real system is one of these — or a mix.

**Why does it matter?** Picking the right pattern is an *architecture* decision: it sets your latency, cost, and how dynamic the workflow can be. Pick wrong and you're slow, expensive, or rigid.

**How does it work?** Three shapes: **sequential** (a line), **parallel** (side by side then merge), **supervisor** (a manager routes). Below, each with a picture, a worked example, and when to choose it.


## 1. Sequential - a pipeline
Agents run **one after another**; each output is the next agent's input. Use when steps **depend** on each other (you can't analyse before you research).

```
  +----------+  notes   +---------+ insights +--------+  memo
  |Researcher| -------> | Analyst | -------> | Writer | -----> done
  +----------+          +---------+          +--------+
```
**Worked example - Razorpay merchant report:** Researcher pulls 6 months of transactions -> raw patterns. Analyst turns patterns into a fraud-risk score -> insights. Writer turns insights into a 1-page client memo -> deliverable. Each step depends on the previous, so a line is the right shape.

**Oracle DBA analogy:** an **ETL pipeline** — Extract → Transform → Load. You can't transform before you extract, so the stages run in order.

**Trade-off:** slowest pattern — every step waits for the one before it.

## 2. Parallel - fan-out, then combine
Several agents work on **independent** sub-tasks; a final agent merges them. Use when sub-tasks don't depend on each other.

```
            +-- Agent A (pricing)    --+
  question -+-- Agent B (logistics)   -+-> Combiner -> one view
            +-- Agent C (competition) --+
```
**Worked example - market-entry study:** pricing, logistics, and competition don't depend on each other, so three researchers study them at once; a Combiner merges the three reports.

**Oracle DBA analogy:** **parallel query** — Oracle splits an independent workload across several worker processes at once, then combines the results. Independent slices run side by side; a final step merges them.

> **Accuracy note:** "parallel" is only *actually* faster if your code runs the calls **concurrently** (e.g. Python threads / async). If you call them in a normal `for` loop, they run one-by-one and you only get the *quality/isolation* benefit, not the speed-up. The lab below shows the real concurrent version.

## 3. Supervisor - a manager delegates
One **supervisor** agent decides *which* specialist to call and *when*, based on the task. Use for **dynamic** workflows where the right next step isn't fixed in advance.

```
                 +--------------+
   task -------> |  Supervisor  | --(reads, decides)--+
                 +--------------+                     |
                  |          |          |             v
            Billing      Technical    Refunds   picks one (or more)
```
**Worked example - support desk:** a ticket arrives; the Supervisor reads it and routes — billing question -> Billing agent, crash report -> Technical agent, refund -> Refunds agent. The path is *decided per ticket*, not fixed.

**Oracle DBA analogy:** the **listener / job scheduler** — it reads each incoming request and routes it to the right service or job, deciding at runtime instead of following a fixed script.

**Trade-off:** most flexible, but the supervisor can route wrong (Section 6).

## Pattern comparison & decision guide
| Pattern | Steps run | Use when | Main trade-off |
|---|---|---|---|
| **Sequential** | One after another | Steps depend on each other | Slowest (waits each step) |
| **Parallel** | At the same time | Sub-tasks are independent | Needs concurrency code + a merge step |
| **Supervisor** | A manager routes | Next step isn't fixed in advance | Routing can go wrong |

**Decision rule:** *Do later steps need earlier results?* Yes -> **sequential**. *Are sub-tasks independent?* Yes -> **parallel**. *Is the right next step unknown until you read the task?* Yes -> **supervisor**.

### AI Architect Perspective
These three patterns are the building blocks; real systems **combine** them. A common production shape:

```
            +------------+
  request ->| Supervisor |  (routes by request type)
            +------------+
                  |  if "deep research" ->
                  v
        +---------------------+   merged
        | Parallel research   | --------> +-----------------+
        | (pricing/logistics/ |          | Sequential:     |
        |  competition)       |          | Analyse -> Write|--> answer
        +---------------------+          +-----------------+
```
The architect's job is to keep it **as simple as it can be**: start sequential, add parallel only where a stage is genuinely independent and latency matters, add a supervisor only when routing is truly dynamic. Every added agent adds cost, latency, and a new failure point.

> **Remember this:** *Sequential = steps depend on each other. Parallel = steps are independent (and you run them concurrently). Supervisor = the next step is decided at runtime.*

### Don't mix these up
- WRONG: "Parallel is always faster."  RIGHT: Only if you actually run the calls concurrently; a plain loop is still sequential timing.
- WRONG: "Supervisor means the supervisor does the work."  RIGHT: The supervisor only *decides who* works; specialists do the work.
- WRONG: "Pick one pattern for the whole system."  RIGHT: Real systems nest patterns inside each other.

### Interview question
**Q (architecture):** "Design the coordination for a research assistant that gathers pricing, logistics and competitor data, then writes a recommendation. Which patterns and why?"
**A:** **Parallel** for the three independent data-gathering agents (run concurrently to cut latency), a **combine** step to merge them, then a short **sequential** analyse-to-write pipeline because writing depends on the merged analysis. Optionally a **supervisor** in front if the request type varies. Justify each: independence -> parallel, dependence -> sequential, dynamic routing -> supervisor.

## Quick Check 3
> **Q:** You summarise customer reviews from 5 cities; the cities don't affect each other; then one final summary. Which pattern(s)?
> **A:** **Parallel** for the 5 independent city summaries (run concurrently), then a **combine** step. A touch of sequential at the end for the merge.

---
# Section 4 - How Agents Communicate (via outputs)

**What is it?** Agents don't share a brain. They communicate by **passing one agent's output text into the next agent's input** — and the *format* of that handoff decides whether your system is reliable or flaky.

**Why does it matter?** The handoff is where multi-agent systems break. Garbled handoffs cause the "telephone game" (Section 6). Clean, structured handoffs are the difference between a demo and a system you can trust.

**How does it work?** The Writer never sees the Researcher's private reasoning — only the **text the Researcher returned**, which *you* place into the Writer's prompt. That string-passing **is** inter-agent communication. The same idea powers **tool outputs**: when an agent calls a tool, the tool's result is a string handed back into the conversation — a standardised, machine-readable output the next step can rely on.


### Free-form vs. structured handoffs
- **Free-form prose** is easy to write but hard to parse — the next agent (or your code) has to *guess* where the useful bits are.
- **Structured** handoffs (bullets, or best, **JSON**) are predictable — the next stage extracts exactly the field it needs. This is the *"tool output standardisation"* idea: a reliable, machine-readable contract between agents.

> **Why this connects to tool use:** a tool call returns a `tool_result` block — a structured string fed back into `messages`. Designing agent handoffs as JSON is the same discipline as designing a clean tool output: a fixed shape both sides agree on.
>
> **Oracle DBA view:** a JSON handoff is like passing a **proper table with named columns** (or a clean staging table) to the next step — every field is where you expect it. A prose handoff is like dumping everything into one **CLOB** and asking the next step to parse it by hand. Structured wins for the same reason a real schema beats free text.

### Message-history isolation
Each `ask_agent` call starts a **fresh `messages` list**. The Analyst does **not** inherit the Researcher's conversation — it only receives the text you hand it. This isolation is a *feature*: it keeps every agent's context clean and cheap. The cost: **information only travels if you pass it** (forget to, and you get "lost context" — Section 6).

### Enterprise Perspective
An insurance claims pipeline uses **JSON handoffs** end to end: an Intake agent returns `{"claim_type": "...", "amount": ..., "flags": [...]}`; a Policy agent reads those exact fields; an Adjudication agent returns a decision object. Because every handoff is a fixed schema, the whole chain is **loggable, testable, and auditable** — a regulator can replay any claim and see exactly what each agent received and returned. Prose handoffs would make that impossible.

> **Remember this:** *Agents communicate by passing outputs, not memories. A structured (JSON) output is a contract the next agent can trust; prose is a guess.*

### Don't mix these up
- WRONG: "Agents share context automatically."  RIGHT: Each agent only knows what you put in its prompt.
- WRONG: "Prose handoffs are fine for long chains."  RIGHT: Prose compounds errors down the chain; JSON keeps each handoff exact.

### Interview question
**Q (advanced):** "Why prefer JSON handoffs over prose in a long agent chain, and how does this relate to tool use?"
**A:** JSON is predictable and machine-parseable, so each stage extracts exactly the field it needs — no guessing, no compounding errors (the telephone effect). It's the same discipline as a tool's structured `tool_result`: a fixed output contract both sides agree on, which makes the chain testable, loggable, and auditable.

## Quick Check 4
> **Q:** Why prefer JSON handoffs over prose in a long agent chain?
> **A:** JSON is **predictable and machine-parseable**, so each stage extracts exactly what it needs. Prose forces guessing and accumulates errors down the chain (the telephone effect).

### See it: a structured (JSON) handoff
Ask the Researcher to output **JSON**, then parse it in code — a reliable machine-readable handoff.

In [ ]:
research_json = ask_agent(
    "You are a market researcher. Reply with ONLY valid JSON: "
    '{"factors": [..5 short strings..], "top_risk": ".."}. No prose.',
    "Zepto expanding to Indian tier-2 cities.",
)
print("Raw handoff:\n", research_json)

# Remove markdown formatting if present
cleaned_json = research_json.strip()
if cleaned_json.startswith('```json') and cleaned_json.endswith('```'):
    cleaned_json = cleaned_json[len('```json'):-len('```')].strip()
elif cleaned_json.startswith('```') and cleaned_json.endswith('```'):
    cleaned_json = cleaned_json[len('```'):-len('```')].strip()

# Parse it - now the next agent can use exact fields, not guess
data = json.loads(cleaned_json)
print("\nParsed top_risk the next agent will receive:", data["top_risk"])

### See it: isolation in action
The second call has no idea what the first said — unless we pass it along. Run this to feel the boundary.

In [ ]:
first  = ask_agent("You are agent A.", "Remember the secret number 42.")
second = ask_agent("You are agent B.", "What secret number did agent A just mention?")
print("Agent B says:\n", second)
# Agent B cannot know - separate memory. Information travels ONLY if you pass it.

---
# Section 5 - Designing Good Roles

**What is it?** The craft of writing the `system` prompt — where specialisation actually happens. A vague role gives a vague agent.

**Why does it matter?** In a team, the *next* agent has to read this output. A predictable role -> a predictable, usable output -> a reliable handoff. Role quality is system quality.

**How does it work?** A strong role has three ingredients:
- **Identity** — "You are a meticulous financial analyst."
- **Scope** — what it should do, and explicitly what it should *not*.
- **Output format** — "Respond as 3 bullets" or "Return JSON with these keys."


### Oracle DBA Perspective
When agents give inconsistent output the next step cannot use, the fix is almost never a bigger model — it is tightening the **output-format** line in each role. This is exactly like enforcing a **schema or a NOT NULL / format constraint**: once the shape is fixed, every consumer downstream can rely on it. Take a vague role, add "Reply with EXACTLY: Verdict: Yes/No. Reason: one sentence," and the output becomes instantly parseable. That small change is often the whole difference between a stalled prototype and a pipeline you can trust.

> **Remember this:** *Identity + Scope + Output format = a role the next agent can trust. The output-format line is the one most beginners skip — and it's the one that makes handoffs reliable.*

### Don't mix these up
- WRONG: "A friendly one-liner role is enough."  RIGHT: Without an explicit output format, the next agent has to guess.
- WRONG: "Tell the agent only what to do."  RIGHT: Also tell it what *not* to do — scope prevents drift.

### Interview question
**Q (intermediate):** "An agent's output is unusable by the next stage. Where do you look first?"
**A:** The **output-format** part of its role. Add an explicit, strict format (bullets or JSON with named keys) so the handoff is predictable — usually fixes it without changing the model.

## Quick Check 5
> **Q:** Two agents, same model, one gives messy output and one gives clean parseable output. What differs?
> **A:** The **role's output-format instruction** — the clean one specifies an exact shape (bullets/JSON); the messy one doesn't.

### See it: vague role vs. strict role
Same question, two roles. The strict role's output is immediately usable by the next agent; the vague one isn't.

In [ ]:
q = "Is Jaipur a good market for Zepto?"
vague  = ask_agent("You help with business stuff.", q)
strict = ask_agent("You are an analyst. Reply with EXACTLY: Verdict: Yes/No. Reason: one sentence.", q)
print("VAGUE:\n", vague[:300])
print("\nSTRICT:\n", strict[:300])

---
# Section 6 - Where Multi-Agent Systems Fail

**What is it?** The five failure modes you'll actually hit in production — each with a symptom and a precise fix. Knowing these is what separates a toy from a system.

**Why does it matter?** Multi-agent bugs are *handoff* and *cost* bugs, not model bugs. Diagnosing by symptom saves hours.


| # | Failure | Symptom | Cause | Fix |
|---|---|---|---|---|
| 1 | **Telephone game** | Details garbled/dropped each handoff | Free-form prose handoffs | Standardise format (bullets/JSON) |
| 2 | **Lost context** | An agent invents info it was never given | Needed data not passed into its prompt | Pass required context explicitly; "use only what's given" |
| 3 | **Runaway cost** | The bill balloons | Many agents = many calls = many tokens; each handoff re-sends prior text | Cheap model (Haiku), concise handoffs, split only when it helps |
| 4 | **Wrong routing** | Supervisor picks the wrong specialist | Specialists' purposes overlap or are vague | Sharpen each specialist's described purpose so they don't overlap |
| 5 | **Over-decomposition** | A simple task split into 5 agents gets slower, costlier, *and* worse | Using multi-agent where one focused agent would do | Only add agents when a step clearly needs its own focus |

> Failures 1 and 2 are *communication* failures (Section 4). Failure 5 is the opposite of Section 1's lesson — the cure can become the disease if you over-apply it.

### Production Considerations
- **Security:** every agent prompt is an injection surface. Treat any text pulled from documents/users as untrusted; tell agents to ignore instructions found *inside* data. Never put API keys in prompts or shared notebooks.
- **Reliability:** wrap JSON parsing in a retry (ask the agent to "return valid JSON only" again on parse failure); add a turn limit to any supervisor loop so it can't run forever.
- **Observability:** log every agent's `system`, input, output, and `usage` tokens. This is your audit trail and your cost dashboard.
- **Cost:** more agents = more tokens; long chains re-send prior text. Use Haiku, keep handoffs concise (JSON helps), and cache stable system prompts where supported.
- **Compliance:** structured handoffs make the chain replayable — essential for regulated industries (finance, insurance, healthcare).

> **Remember this:** *Multi-agent bugs are handoff bugs and cost bugs. Diagnose by symptom: garbled = telephone, invented = lost context, expensive = too many tokens, mis-sent = bad routing, slow-and-worse = over-split.*

### Interview question
**Q (Oracle DBA angle):** "A Writer agent keeps inventing sales numbers the Researcher never produced. What's wrong and how do you fix it without a bigger model?"
**A:** **Lost context** — the real figures weren't passed into the Writer's prompt. Fix: pass the Researcher's structured output explicitly into the Writer, and instruct the Writer to use *only* the data it's given (no invention). A JSON handoff makes this exact and testable.

## Quick Check 6
> **Q:** Your Writer keeps inventing sales numbers the Researcher never found. Which failure, and the fix?
> **A:** **Lost context** — the Writer didn't receive the real figures. Fix: pass the Researcher's output explicitly and tell the Writer to use only what it's given.

### See it: cost grows with every agent
Each agent call costs tokens. Print usage to *feel* why long chains get expensive.

In [ ]:
r = client.messages.create(model=MODEL, max_tokens=200,
    system="You are a researcher. 3 bullets.",
    messages=[{"role": "user", "content": "Factors for entering Jaipur?"}])
print("One agent call used:", r.usage.input_tokens, "input /", r.usage.output_tokens, "output tokens")
print("A 5-agent chain roughly multiplies this - and re-sends prior text at each handoff.")

---
# Lab - Build a Real Multi-Agent Pipeline with Claude

**Objective:** build all three patterns end to end against the live Claude API, using the `ask_agent` helper you already have. Run each cell, read the output, then try the challenge.

> Prereq: you ran the two setup cells and the `ask_agent` cell at the top. Cost is tiny (Haiku, short prompts).

### Lab 1 - Sequential pipeline (Researcher -> Analyst -> Writer)
**Objective:** see how each agent's output becomes the next one's input.
**Expected result:** three clearly different outputs — raw factors, then a risk read, then a short memo built from them.

In [ ]:
# SEQUENTIAL: each step feeds the next (the output IS the handoff)
topic = "Zepto expanding 10-minute delivery to Jaipur"

notes = ask_agent(
    "You are a market researcher. Output 5 short factual bullets only.",
    f"Research: {topic}")

insights = ask_agent(
    "You are a risk analyst. Given research bullets, output: Risk level (Low/Med/High) + 2 reasons.",
    f"Here is the research:\n{notes}\n\nAssess the risk.")

memo = ask_agent(
    "You are a business writer. Write a 3-sentence recommendation for an executive.",
    f"Research:\n{notes}\n\nRisk assessment:\n{insights}\n\nWrite the memo.")

print("1) RESEARCH:\n", notes)
print("\n2) RISK:\n", insights)
print("\n3) MEMO:\n", memo)

**Try this:** add a 4th agent — an *Editor* — that takes `memo` and tightens it to 2 sentences. Notice you only pass it the `memo`, not the earlier steps.

### Lab 2 - Parallel (concurrent) fan-out, then combine
**Objective:** run three independent research agents **at the same time** with threads, then merge — the *real* speed-up version from Section 3.
**Expected result:** three angles gathered concurrently, then one combined view.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

angles = {
    "pricing":     "You research PRICING only. 3 bullets.",
    "logistics":   "You research LOGISTICS only. 3 bullets.",
    "competition": "You research COMPETITION only. 3 bullets.",
}
task = "Zepto entering Jaipur - your angle only."

# Run all three agents CONCURRENTLY (this is what makes 'parallel' actually faster)
with ThreadPoolExecutor(max_workers=3) as pool:
    results = dict(zip(angles, pool.map(lambda r: ask_agent(r, task), angles.values())))

merged = "\n\n".join(f"[{k.upper()}]\n{v}" for k, v in results.items())
combined = ask_agent(
    "You are a strategy lead. Merge these three reports into one 4-bullet view.",
    f"Reports:\n{merged}")
print("COMBINED VIEW:\n", combined)

**Try this:** time it. Wrap the `with` block in `import time; t=time.time()` ... `print(time.time()-t)` and compare against calling the three `ask_agent`s in a plain loop. The concurrent version should finish in roughly the time of the slowest single call.

### Lab 3 - Supervisor routing
**Objective:** let one agent *decide* which specialist handles a ticket, then call that specialist.
**Expected result:** the Supervisor returns a single category; your code routes to the matching agent.

In [ ]:
specialists = {
    "billing":   "You are a billing support agent. Answer briefly and helpfully.",
    "technical": "You are a technical support agent. Answer briefly and helpfully.",
    "refund":    "You are a refunds agent. Answer briefly and explain the refund step.",
}

def handle_ticket(ticket):
    # 1) Supervisor decides the route (constrained to one word we can trust)
    route = ask_agent(
        "You are a support supervisor. Reply with EXACTLY ONE word: billing, technical, or refund.",
        ticket, max_tokens=10).strip().lower()
    route = route if route in specialists else "technical"   # safe fallback
    # 2) Hand the ticket to the chosen specialist
    answer = ask_agent(specialists[route], ticket)
    return route, answer

for t in ["My card was charged twice!", "The app crashes on checkout.", "I want my money back for order #91."]:
    route, answer = handle_ticket(t)
    print(f"TICKET: {t}\n  -> routed to [{route}]\n  {answer[:160]}\n")

**Try this:** add a 4th specialist (`"account"`) and a ticket that should route to it. Notice failure mode #4 (wrong routing) if two specialists' descriptions overlap — sharpen them.

---
# Architecture - A Production Multi-Agent System

Putting it together: a **Customer Insight Engine** that a retailer runs on incoming feedback. It nests all three patterns.

```
                         +-------------------+
   feedback item ------> |    SUPERVISOR     |  routes by type
                         +-------------------+
                            |            |
              "needs research"        "simple FAQ"
                            v            v
            +-----------------------+   +-----------+
            |  PARALLEL (threads)   |   | FAQ agent |--> answer
            |  pricing | logistics  |   +-----------+
            |  | sentiment          |
            +-----------------------+
                       | merged JSON
                       v
            +-----------------------+
            | SEQUENTIAL pipeline   |
            | Analyst -> Writer     |--> insight report
            +-----------------------+
```

**Component responsibilities**
- **Supervisor:** classify each item; route to FAQ (cheap path) or the research path. *Failure point:* wrong routing -> keep specialist purposes distinct.
- **Parallel researchers:** independent angles, run concurrently, each returns **JSON**. *Failure point:* one agent stalls -> set timeouts / fallbacks.
- **Combiner:** merges JSON into one object. *Failure point:* malformed JSON -> validate + retry.
- **Sequential Analyst -> Writer:** depends on merged data; produces the final report.

**Data flow:** feedback -> supervisor label -> (FAQ answer) OR (parallel JSON -> merged JSON -> analysis -> report). Every handoff is a logged string/JSON.

**Scaling · Security · Cost**
- **Scale:** parallel stage scales with worker threads; cap concurrency to control spend and rate limits.
- **Security:** feedback is untrusted input — instruct agents to ignore embedded instructions (prompt-injection defence); keep keys in secrets.
- **Cost:** route cheap items to the single FAQ agent; reserve the multi-agent path for items that need it; use Haiku and concise JSON handoffs.

**The architect's tradeoff:** every agent you add buys quality but costs latency, tokens, and a new failure point. Start with the simplest pattern that works and grow only where a stage clearly earns it.

---
# Mini Project - "Market-Entry Brief" Multi-Agent Assistant

**Business use case:** a consulting team wants a one-click brief on whether a delivery company should enter a new city. Today an analyst spends a day; you'll build a Claude team that drafts it in seconds for a human to review.

**Architecture (build it with what you learned):**
```
  city + company
        |
        v
  PARALLEL (3 threads, JSON out):  Pricing | Logistics | Competition
        |  merged JSON
        v
  SEQUENTIAL:  Risk Analyst (risk level + reasons)  ->  Writer (1-page brief)
        |
        v
  one Market-Entry Brief (with a 'confidence' line)
```

**Claude build steps**
1. Reuse `ask_agent`. Write 3 research roles, each returning **JSON** (`{"points":[...]}`) for its angle.
2. Run the three **concurrently** with `ThreadPoolExecutor` (Lab 2 pattern).
3. Merge the three JSON blobs into one dict; pass it to a **Risk Analyst** role -> risk level + 2 reasons.
4. Pass research + risk to a **Writer** role -> a 1-page brief ending in "Confidence: Low/Med/High".
5. Print total `usage` tokens across all calls so you can show the cost.

**Definition of done**
- Runs end to end on `("Zepto", "Jaipur")` and prints a readable brief.
- The three research calls run concurrently (not in a plain loop).
- At least one handoff is **JSON** that you `json.loads` and use by field.
- You can state which **failure mode** each design choice guards against (telephone -> JSON; lost context -> explicit passing; cost -> Haiku + concise handoffs).

**Stretch:** add a **Supervisor** in front that skips the whole pipeline and returns a one-line answer for trivial questions ("is Jaipur in India?") — proving you avoid *over-decomposition* (failure #5).

---
# Revision Suite

## Session Summary
You learned why a **team of focused agents** beats one overloaded agent (attention, not knowledge, is the bottleneck), what an **agent** really is (role + own memory + input/output), the **three coordination patterns** (sequential / parallel / supervisor) and how to pick them, how agents **communicate by passing outputs** (prose vs. structured JSON, the same discipline as tool outputs), how to **design strong roles** (identity + scope + output format), and the **five failure modes** with their fixes. You built all three patterns live and designed a production architecture and mini project.

## What You Learned Today - you can now...
- **Explain** context dilution + instruction blur, and why splitting work raises quality.
- **Build** an agent from a role string with one `ask_agent` helper.
- **Implement** sequential, concurrent-parallel, and supervisor patterns against the live Claude API.
- **Design** reliable JSON handoffs and strong role prompts.
- **Diagnose** the five multi-agent failures by symptom and apply the fix.
- **Architect** a nested multi-agent system and reason about its cost, scale, and security.

## AI Architect Cheat Sheet
**Agent** = `system` role + own `messages` + input->output. Specialisation lives in `system`.

**Patterns**
| Pattern | Use when | Trade-off |
|---|---|---|
| Sequential | Steps depend on each other | Slowest |
| Parallel (concurrent) | Sub-tasks independent | Needs threads/async + merge |
| Supervisor | Next step decided at runtime | Routing can go wrong |

**Decision rule:** later step needs earlier result? -> sequential. Independent? -> parallel. Unknown next step? -> supervisor.

**Handoffs:** prose = guess; **JSON = contract**. Same idea as a tool's `tool_result`.

**Strong role = Identity + Scope + Output format.**

**Failure -> fix:** garbled = telephone -> JSON · invented = lost context -> pass it explicitly · expensive = too many tokens -> Haiku + concise · mis-sent = bad routing -> sharpen purposes · slow-and-worse = over-split -> use one agent.

**Claude API quick-ref**
```python
reply = client.messages.create(model=MODEL, max_tokens=400,
    system="<role>", messages=[{"role":"user","content":"<task>"}])
reply.content[0].text     # the answer
reply.usage               # input/output token counts (cost)
reply.stop_reason         # 'end_turn' done · 'tool_use' wants a tool
```

## 5-Minute Revision Guide
1. **Why teams win:** one prompt with many jobs -> attention diluted, rules blurred. Split -> each step gets full focus.
2. **An agent** = role + own memory + I/O. Change behaviour by changing only `system`.
3. **Three patterns:** sequential (dependent line), parallel (independent, run concurrently, then merge), supervisor (runtime routing). They nest.
4. **Communication** = passing outputs. JSON handoffs = reliable contract; prose = telephone game. Same as tool outputs.
5. **Roles:** identity + scope + output format. The output-format line makes handoffs usable.
6. **Five failures:** telephone, lost context, runaway cost, wrong routing, over-decomposition — know symptom + fix.
7. **Architect's law:** every added agent buys quality but costs latency, tokens, and a failure point. Start simplest.

## Interview Preparation Notes
- **"Why does a multi-agent team beat one agent?"** Attention, not knowledge: one prompt with many jobs causes context dilution and instruction blur; focused agents get full focus + clean handoffs + easier debugging.
- **"Name the three patterns and when to use each."** Sequential (steps depend), parallel (independent, run concurrently then merge), supervisor (dynamic routing). Real systems combine them.
- **"How do agents communicate?"** By passing one agent's output into the next's input. Prefer structured JSON — predictable, parseable, auditable — the same discipline as tool `tool_result` outputs.
- **"Design a research assistant's coordination."** Parallel data-gathering -> combine -> sequential analyse/write, supervisor in front if request types vary. Justify each by dependence/independence/dynamism.
- **"A Writer invents numbers the Researcher never produced — fix it?"** Lost context: pass the Researcher's structured output explicitly and instruct 'use only given data'. JSON makes it exact.
- **"How do you control multi-agent cost?"** Fewer agents where possible, Haiku, concise JSON handoffs, route cheap items to a single agent, cap concurrency, log `usage`.

## Assignment
- **Beginner:** rewrite the `ask_agent` "cautious vs. bold" demo with two *new* roles (e.g. lawyer vs. marketer) on a question of your choice.
- **Intermediate:** build a 3-step sequential pipeline for a different domain (e.g. resume -> screener -> interviewer questions). Use a JSON handoff at one step.
- **Advanced:** convert the parallel lab to time concurrent vs. looped execution and report the speed-up, plus total tokens.
- **Project:** complete the **Market-Entry Brief** mini project end to end (concurrent research + JSON handoff + risk + writer), and write 3 sentences on which failure mode each design choice guards against.

## Assessment

**Part A - 10 Multiple Choice**
1. The main reason one overloaded agent underperforms a team is: (a) it knows less (b) context dilution + instruction blur (c) a smaller model (d) no internet.
2. An "agent" in this session is best defined as: (a) a fine-tuned model (b) role + own memory + input/output (c) a database (d) a tool.
3. Specialisation between agents primarily comes from: (a) the model name (b) the `system` prompt (c) max_tokens (d) temperature.
4. Steps that depend on each other should be wired as: (a) parallel (b) supervisor (c) sequential (d) random.
5. Independent sub-tasks that you want faster should be: (a) sequential (b) run concurrently (parallel) (c) one big prompt (d) supervised one at a time.
6. A supervisor agent's job is to: (a) do all the work (b) decide which specialist runs (c) store memory (d) parse JSON.
7. Agents communicate by: (a) sharing memory automatically (b) passing one's output into the next's input (c) a global variable (d) the model's training data.
8. The most reliable handoff format for long chains is: (a) free-form prose (b) JSON/structured (c) an image (d) emojis.
9. "An agent invents data it was never given" is which failure: (a) telephone (b) lost context (c) runaway cost (d) wrong routing.
10. Splitting a tiny single-skill task into 5 agents is: (a) always better (b) over-decomposition (c) parallelism (d) supervision.

**Part B - 5 Short Answer**
1. In one sentence, why does splitting a multi-job prompt raise quality?
2. What three things make up an "agent" here?
3. Give the decision rule for choosing sequential vs. parallel vs. supervisor.
4. Why are JSON handoffs better than prose, and how does this relate to tool outputs?
5. Name two ways to control the cost of a multi-agent system.

**Part C - 3 Scenarios**
1. A 6-agent chain's output is subtly wrong and details drift between steps. Diagnose and fix.
2. A client's supervisor keeps sending billing tickets to the technical agent. Cause and fix?
3. You must gather pricing, logistics, and competitor data then write a recommendation, as fast as possible. Describe the coordination design.

## Answer Key
**Part A:** 1-b, 2-b, 3-b, 4-c, 5-b, 6-b, 7-b, 8-b, 9-b, 10-b.

**Part B:**
1. Each step gets the model's full attention instead of competing instructions diluting focus.
2. A role (`system` prompt), its own memory (separate `messages`), and an input->output.
3. Later step needs earlier result -> sequential; sub-tasks independent -> parallel (concurrent); next step unknown until runtime -> supervisor.
4. JSON is predictable and machine-parseable so each stage extracts exact fields with no guessing or compounding errors; it's the same contract idea as a tool's structured `tool_result`.
5. Any two: use Haiku, keep handoffs concise/JSON, fewer agents, route cheap items to one agent, cap concurrency, log `usage`.

**Part C:**
1. **Telephone game** from prose handoffs — switch handoffs to JSON/structured so each stage passes exact fields.
2. **Wrong routing** — billing and technical specialist descriptions overlap or are vague; sharpen each specialist's purpose so they're distinct, and constrain the supervisor's output.
3. **Parallel (concurrent)** pricing/logistics/competition agents -> **combine** -> short **sequential** analyse->write pipeline; optional supervisor in front if request types vary.

---
### End of Weekend 9 - Multi-Agent Systems
You can now explain, build, and architect teams of agents. Next: persistent memory and MCP — giving these agents long-term knowledge and real tools.